# Retrieval backend trên Kaggle — mapping, embedding, OCR Elasticsearch, foreground logs
Bật Internet; chọn GPU nếu chạy visual encoder. Notebook tải ASR/OCR/caption JSON, map_keyframes CSV, embedding ảnh/ASR/caption theo range inclusive đã cấu hình trong DATASET_RANGES. Không tải ảnh keyframe. Mỗi source có root folder ID riêng; notebook tự đặt thư mục đích và bung ZIP embedding theo cấu trúc video/segment (`keyframe/<DATASET>/<video_id>/<frame>.npy`, `asr/<DATASET>/<video_id>/<segment_number>.npy`, `caption/<DATASET>/<video_id>/<segment_number>.npy`). Host ghép từng vector theo segment ordinal; segment thiếu/hỏng bị bỏ riêng, không làm mất video; không embed lại ASR/caption. Notebook không tự chạy các model extraction. Điền nguồn thật trong cell cấu hình. Chạy cell cuối liên tục để giữ supervisor và xem log; Kaggle vẫn có thể kết thúc phiên theo quota/session limits.

Nguồn: [Kaggle notebooks](https://www.kaggle.com/docs/notebooks), [Elasticsearch archive / non-root](https://www.elastic.co/docs/deploy-manage/deploy/self-managed/install-elasticsearch-from-archive-on-linux-macos).


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil
WORK = Path('/kaggle/working')
REPO = WORK / 'event-retrieval-system'
REPO_URL = 'https://github.com/iamtro-2006/event-retrieval-system.git'
# Chọn branch/commit chứa thay đổi mới, hoặc attach repo vào /kaggle/input.
REPO_REF = 'vbs-dev'  # Branch chứa scripts/build_segment_data.py và configs/app.yaml
DATA = REPO / 'data'
RCLONE_CONFIG_SOURCE = Path('/kaggle/input/datasets/nguyennhattan121106/rclone-pro-pro/rclone.conf')
DATASETS = ['v3c']
# Hai đầu được lấy, cùng quy ước range trong host_server.ipynb.
DATASET_RANGES = {'v3c': (1, 100)}  # Inclusive: V3C ID 00001–00100; LHE ví dụ (1, 75)
QWEN_MODEL = 'Qwen/Qwen3-VL-Embedding-2B'
SEGMENT_BATCH_SIZE = 8
SEGMENT_SOURCES = ['asr', 'caption']  # Phải khớp những source đã embed.
# Mỗi source có root folder ID riêng; folder trỏ thẳng vào các file nguồn.
# Embedding folder chứa <video_id>.zip; trong ZIP có <video_id>/<ordinal>.npy.
# Ảnh keyframe gốc không tải về Kaggle.
DRIVE_FOLDER_IDS = {
    'v3c': {
        'asr': '',
        'caption': '',
        'ocr': '',
        'map_keyframes': '',
        'keyframe_embeddings': '',
        'asr_embeddings': '',
        'caption_embeddings': '',
    },
    # 'lhe': {'asr':'', 'caption':'', 'ocr':'', 'map_keyframes':'',
    #         'keyframe_embeddings':'', 'asr_embeddings':'', 'caption_embeddings':''},
}

# Visual and segment embeddings use Qwen vLLM.
VISUAL_ENGINE = 'vllm'
VLLM_COMMAND = 'auto'  # Install isolated runtime; use None for an existing remote endpoint.
# Example: ['python','-m','vllm.entrypoints.openai.api_server','--model',
# 'Qwen/Qwen3-VL-Embedding-2B','--runner','pooling','--port','8002']
# Obtain optional NGROK_AUTHTOKEN from Kaggle Secrets below. Never paste credentials here.


In [ ]:
import urllib.request
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', REPO_REF,
                    REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', REPO_REF, 'FETCH_HEAD'], check=True)
BACKEND = REPO / 'backend'
required_files = [BACKEND/'scripts/build_segment_data.py', BACKEND/'configs/app.yaml',
                  BACKEND/'scripts/kaggle_host.py']
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f'Repo {REPO_REF} is incomplete; missing: {missing_files}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'numpy','pandas','faiss-cpu','fastapi','uvicorn','python-multipart','httpx','PyYAML',
    'python-dotenv','elasticsearch>=8.18,<9','sentence-transformers','Pillow','huggingface_hub',
    'pyngrok','open_clip_torch','timm','einops','uv'], check=True)
os.chdir(BACKEND)
sys.path.insert(0, str(BACKEND))

rclone = shutil.which('rclone')
if not rclone:
    import zipfile
    archive=WORK/'rclone-v1.75.2-linux-amd64.zip'
    urllib.request.urlretrieve('https://github.com/rclone/rclone/releases/download/v1.75.2/rclone-v1.75.2-linux-amd64.zip',archive)
    with zipfile.ZipFile(archive) as z:
        member=next(x for x in z.namelist() if x.endswith('/rclone'))
        rclone_bin=WORK/'rclone'; rclone_bin.write_bytes(z.read(member)); rclone_bin.chmod(0o755)
    rclone=str(rclone_bin)
RCLONE_CONFIG = WORK/'rclone.conf'
RCLONE_CONFIG_READY = False
try:
    from kaggle_secrets import UserSecretsClient
    config_text=UserSecretsClient().get_secret('RCLONE_CONFIG')
except Exception:
    config_text=None
if config_text:
    RCLONE_CONFIG.write_text(config_text,encoding='utf-8'); RCLONE_CONFIG_READY=True
elif RCLONE_CONFIG_SOURCE.is_file():
    shutil.copy2(RCLONE_CONFIG_SOURCE,RCLONE_CONFIG); RCLONE_CONFIG_READY=True
if RCLONE_CONFIG_READY:
    RCLONE_CONFIG.chmod(0o600)
    subprocess.run([rclone,'--config',str(RCLONE_CONFIG),'listremotes'],check=True)
else:
    raise FileNotFoundError(f'Rclone config not found: {RCLONE_CONFIG_SOURCE}; attach it as Kaggle Input')

In [ ]:
import zipfile
from collections import Counter


def video_name(dataset, number):
    return f'LHE{number:02d}' if dataset.lower() == 'lhe' else f'{number:05d}'


def download_drive_folder(folder_id, target, video_ids, suffix, allow_missing=False):
    if not folder_id:
        if allow_missing:
            print(f'{target}: no source folder ID; skip optional ASR', flush=True)
            return []
        raise ValueError(f'Missing Drive root folder ID for {target}')
    target.mkdir(parents=True, exist_ok=True)
    listing = subprocess.run(
        [rclone, '--config', str(RCLONE_CONFIG), '--drive-root-folder-id', folder_id,
         'lsf', 'iamtro:', '--files-only', '--format', 'p'],
        check=True, capture_output=True, text=True)
    files = [line.strip() for line in listing.stdout.splitlines() if line.strip()]
    expected = {f'{video_id}{suffix}' for video_id in video_ids}
    matches = [name for name in files if name in expected]
    duplicates = sorted(name for name, count in Counter(matches).items() if count > 1)
    available = set(matches)
    missing = sorted(expected - available)
    if duplicates:
        raise RuntimeError(f'Duplicate source files in {target}: {duplicates}')
    if missing and not allow_missing:
        raise FileNotFoundError(f'{target}: missing {len(missing)} {suffix} files in range; examples={missing[:10]}')
    if missing:
        print(f'{target}: skip {len(missing)} missing optional ASR files: {missing[:10]}', flush=True)
    for name in sorted(available):
        subprocess.run(
            [rclone, '--config', str(RCLONE_CONFIG), '--drive-root-folder-id', folder_id,
             'copyto', f'iamtro:{name}', str(target/name)], check=True)
    if not available and allow_missing:
        print(f'{target}: no matching ASR files in selected range; skip', flush=True)
    else:
        print(f'{target}: downloaded {len(available)}/{len(expected)} files for selected range', flush=True)
    return sorted(available)


def unpack_video_embedding_zips(folder, video_ids, allow_missing=False):
    archives = sorted(folder.glob('*.zip'))
    available = {archive.stem for archive in archives}
    missing = sorted(set(video_ids) - available)
    if missing and not allow_missing:
        raise FileNotFoundError(f'{folder}: missing {len(missing)} video ZIPs in range; examples={missing[:10]}')
    if missing:
        print(f'{folder}: skip {len(missing)} missing optional ASR ZIPs: {missing[:10]}', flush=True)
    if not archives and allow_missing:
        print(f'{folder}: no matching ASR ZIPs; skip', flush=True)
        return
    for archive in archives:
        target = folder
        with zipfile.ZipFile(archive) as bundle:
            bad = bundle.testzip()
            if bad:
                raise ValueError(f'Corrupt embedding archive {archive}: {bad}')
            for member in bundle.namelist():
                rel = Path(member)
                if rel.is_absolute() or '..' in rel.parts or '\\' in member:
                    raise ValueError(f'Unsafe archive member: {member}')
                if not (target / rel).resolve().is_relative_to(target.resolve()):
                    raise ValueError(f'Archive member escapes target: {member}')
                if rel.parts and rel.parts[0] != archive.stem:
                    raise ValueError(f'ZIP {archive.name} must contain {archive.stem}/...; found {member}')
            bundle.extractall(target)
        archive.unlink()
    print(f'{folder}: expanded {len(archives)} video ZIPs', flush=True)


RAW_SOURCE_DIRS = {'asr':'asr', 'caption':'captions', 'ocr':'ocr', 'map_keyframes':'map_keyframes'}
for dataset in DATASETS:
    if dataset not in DATASET_RANGES:
        raise KeyError(f'Missing range for dataset {dataset}')
    start, end = DATASET_RANGES[dataset]
    max_id = 99 if dataset.lower() == 'lhe' else 99999
    if type(start) is not int or type(end) is not int or not 1 <= start <= end <= max_id:
        raise ValueError(f'Invalid inclusive range for {dataset}: {(start, end)}')
    selected_ids = [video_name(dataset, number) for number in range(start, end+1)]
    ids = DRIVE_FOLDER_IDS[dataset]
    upper = dataset.upper()
    for source, directory in RAW_SOURCE_DIRS.items():
        target = DATA/directory/upper
        if target.exists():
            shutil.rmtree(target)
        download_drive_folder(ids[source], target, selected_ids, '.csv' if source=='map_keyframes' else '.json', allow_missing=(source=='asr'))
    for source in ('keyframe', 'asr', 'caption'):
        folder = DATA/'embeddings'/'embeddings_qwen3_vl_embedding_2b_vllm'/source/upper
        if folder.exists():
            shutil.rmtree(folder)
        download_key = f'{source}_embeddings'
        optional_asr = source == 'asr'
        download_drive_folder(ids[download_key], folder, selected_ids, '.zip', allow_missing=optional_asr)
        unpack_video_embedding_zips(folder, selected_ids, allow_missing=optional_asr)
print('Downloaded range-limited sources.')


## Cấu hình dataset và build mapping/index
Cần map_keyframes, embedding keyframe/ASR/caption, ASR/caption JSON, OCR JSON. Không tải ảnh keyframe; visual FAISS được build từ embedding ảnh và map. Caption/ASR cùng visual dùng Qwen3-VL-Embedding-2B qua vLLM. Không có dữ liệu thật thì dừng rõ lỗi; không fallback sang mock. Text model chung cho ASR/caption có thể đổi; đổi model phải rebuild cả hai index.


In [ ]:
if VISUAL_ENGINE == 'vllm' and VLLM_COMMAND == 'auto':
    VLLM_VENV = WORK / 'ers-vllm-venv'
    if not (VLLM_VENV/'bin/vllm').exists():
        subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', str(VLLM_VENV)], check=True)
        subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(VLLM_VENV/'bin/python'),
                        '--torch-backend=auto', '-r', str(BACKEND/'requirements-vllm.txt')], check=True)
    VLLM_COMMAND = [str(VLLM_VENV/'bin/vllm'), 'serve', 'Qwen/Qwen3-VL-Embedding-2B',
                    '--runner', 'pooling', '--max-model-len', '8192', '--host', '127.0.0.1', '--port', '8002']
    os.environ['QWEN_VLLM_URL'] = 'http://127.0.0.1:8002/v1'
elif VLLM_COMMAND == 'auto':
    VLLM_COMMAND = None


In [ ]:
import yaml
import numpy as np
APP_CONFIG = BACKEND / 'configs/app.yaml'
cfg = yaml.safe_load(APP_CONFIG.read_text())
for key in list(cfg['datasets']['available']):
    if key not in DATASETS:
        del cfg['datasets']['available'][key]
cfg['datasets']['default'] = DATASETS[0]
cfg['segment_data'] = {'data_root': str(DATA)}
cfg['asr'] = None
# No translation API key is required for the notebook's default multilingual search.
cfg.setdefault('translate', {})['enabled_default'] = False
for dataset, spec in cfg['datasets']['available'].items():
    spec.update(root=str(DATA), keyframes_root=str(DATA/'keyframes'),
                videos_root=str(DATA/'videos'), map_keyframes_root=str(DATA/'map_keyframes'))
    # Paths/engine suffixes must match the downloaded visual index artifacts.
    for model in cfg['semantic']['models']:
        model.setdefault('model_extra', {})['engine'] = VISUAL_ENGINE
# Fail visibly before hosting if downloaded visual artifacts are absent.
for dataset, spec in cfg['datasets']['available'].items():
    for model in cfg['semantic']['models']:
        if not model.get('enabled', True):
            continue
        override = spec['semantic_models'].get(model['model_key'], {})
        directory = override.get('index_dir')
        if directory:
            directory = Path(directory)
            if not directory.is_absolute():
                directory = BACKEND / directory
            if not (directory/'keyframe.faiss').is_file():
                # Reuse the repository's visual index builder if only raw embeddings were downloaded.
                from src.retrieval.indexer.faiss.build_pipeline import BuildFaissIndexPipeline
                embeddings = DATA/'embeddings'/f'embeddings_qwen3_vl_embedding_2b_{VISUAL_ENGINE}'/'keyframe'
                assert embeddings.exists(), f'Download a visual index or raw embeddings into {embeddings}'
                indexing = yaml.safe_load((BACKEND/'configs/indexing.yaml').read_text())
                build = {**indexing, **indexing['datasets']['available'][dataset],
                         'keyframes_root':str(DATA/'keyframes'),
                         'map_keyframes_root':str(DATA/'map_keyframes'),
                         'embeddings_root':str(embeddings), 'collection':dataset,
                         'index':{**indexing['index'], 'output_dir':str(directory)}}
                print('Building visual FAISS index:', dataset, directory, flush=True)
                BuildFaissIndexPipeline(build).run()
            for filename in ('keyframe.faiss', 'metadata.csv'):
                assert (directory/filename).is_file(), f'Download visual artifact: {directory/filename}'
            cache_path = directory/'vectors_fp32.npy'
            if not cache_path.is_file():
                subprocess.run([sys.executable,'-u','-m','src.retrieval.indexer.faiss.vector_cache',
                                '--index-path',str(directory/'keyframe.faiss'),'--output',str(cache_path),
                                '--dtype','float32','--batch-size','50000'],cwd=BACKEND,check=True)
# Write runtime config (not committed); scripts and API use it through CONFIG_PATH.
RUNTIME_CONFIG = BACKEND/'configs/kaggle.runtime.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
os.environ['CONFIG_PATH'] = str(RUNTIME_CONFIG)
os.environ['QWEN_EMBEDDING_ENGINE'] = 'vllm'
os.environ.setdefault('QWEN_VLLM_URL', 'http://127.0.0.1:8002/v1')
if VISUAL_ENGINE != 'vllm':
    raise RuntimeError('Visual and text indexes must use Qwen vLLM')
from src.retrieval.segment_data import build_maps, load_segment_files, SegmentMaps
for dataset in DATASETS:
    build_maps(DATA,dataset.lower())
    for folder in ('captions','ocr'):
        assert list((DATA/folder/dataset.upper()).glob('*.json')), f'Missing {folder}/{dataset}'
    if 'asr' in SEGMENT_SOURCES and not list((DATA/'asr'/dataset.upper()).glob('*.json')):
        print(f'asr/{dataset}: no JSON files; ASR index will be skipped',flush=True)
    assert list((DATA/'map_keyframes'/dataset.upper()).glob('*.csv')), f'Missing map_keyframes/{dataset}'
    for source in SEGMENT_SOURCES:
        segment_dir = DATA/'asr' if source=='asr' else DATA/'captions'
        json_files = list((segment_dir/dataset.upper()).glob('*.json'))
        source_files=dict(load_segment_files(DATA,dataset.lower(),source))
        if source=='caption':
            all_records=[record for records in source_files.values() for record in records]
            maps=SegmentMaps(DATA,dataset.lower(),'caption',all_records)
            source_files={video_id:[record for record in records if maps.forward.get(record['segment_id'])]
                          for video_id,records in source_files.items()}
        for json_path in json_files:
            records=source_files.get(json_path.stem,[])
            vector_dir=DATA/'embeddings'/'embeddings_qwen3_vl_embedding_2b_vllm'/source/dataset.upper()/json_path.stem
            missing=[]
            for record in records:
                ordinal=int(record['segment_id'].rsplit(':',1)[-1])
                vector_path=vector_dir/f'{ordinal:06d}.npy'
                if not vector_path.is_file():
                    missing.append(record['segment_id'])
                    continue
                vector=np.load(vector_path,mmap_mode='r',allow_pickle=False)
                if vector.shape!=(2048,): missing.append(record['segment_id'])
            if missing: print(f'{source}/{json_path.stem}: will skip {len(missing)} missing/invalid segments',flush=True)
print('Inputs ready; the host supervisor will start Qwen vLLM, build ASR/caption FAISS, seed OCR, and start the API.')

## Cài Elasticsearch local (không cần Docker)
Tải archive chính thức và kiểm checksum. Cell cuối chạy Elasticsearch bằng tài khoản không phải root, seed OCR với `--recreate`, và giám sát toàn bộ tiến trình. Elasticsearch chỉ bind localhost.


In [ ]:
import hashlib
ES_VERSION = '8.18.0'
ES_HOME = WORK / f'elasticsearch-{ES_VERSION}'
if not ES_HOME.exists():
    name = f'elasticsearch-{ES_VERSION}-linux-x86_64.tar.gz'
    archive = WORK/name
    url = 'https://artifacts.elastic.co/downloads/elasticsearch/'+name
    urllib.request.urlretrieve(url, archive)
    expected = urllib.request.urlopen(url+'.sha512').read().decode().split()[0]
    digest = hashlib.sha512()
    with archive.open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    assert digest.hexdigest() == expected, 'Elasticsearch checksum mismatch'
    with tarfile.open(archive) as bundle:
        bundle.extractall(WORK, filter='data')
(ES_HOME/'config/elasticsearch.yml').write_text("""cluster.name: ers-kaggle
node.name: notebook
network.host: 127.0.0.1
http.port: 9200
discovery.type: single-node
xpack.security.enabled: false
xpack.ml.enabled: false
""")
print('ES installed:', ES_HOME)


## Chạy foreground — giữ cell này đang chạy
Log FastAPI/access, lỗi startup/search, log ES và heartbeat đều hiện trong cell. Dừng cell sẽ dọn các tiến trình do supervisor tạo. Log không ngăn Kaggle thu hồi phiên/quota hay OOM; cần lưu output/index ra nguồn bền vững. Backend không tự đăng nhập Codex trên Kaggle: agent cần CLI/login hợp lệ ở host chạy bridge; các endpoint search thường vẫn hoạt động độc lập.

Nếu dùng vLLM, cài theo `backend/requirements-vllm.txt` trong môi trường tương thích rồi điền `VLLM_COMMAND`, hoặc cung cấp endpoint qua `QWEN_VLLM_URL`. Không dùng index được tạo bởi engine khác.


In [ ]:
from scripts.kaggle_host import serve
ngrok_token = None
try:
    from kaggle_secrets import UserSecretsClient
    ngrok_token = UserSecretsClient().get_secret('NGROK_AUTHTOKEN')
except Exception:
    print('No ngrok secret: API available inside notebook at http://127.0.0.1:8000', flush=True)
# Intentionally await the server IN THIS CELL: no detached API subprocess, no reload.
await serve({'backend':str(BACKEND), 'app_config':str(RUNTIME_CONFIG),
             'es_home':str(ES_HOME), 'datasets':DATASETS, 'port':8000,
             'vllm_command':VLLM_COMMAND, 'ngrok_token':ngrok_token,
             'build_segment_indexes':True, 'data_root':str(DATA),
             'segment_embeddings_root':str(DATA/'embeddings'/'embeddings_qwen3_vl_embedding_2b_vllm'),
             'segment_model':QWEN_MODEL, 'segment_batch_size':SEGMENT_BATCH_SIZE,
             'segment_sources':SEGMENT_SOURCES})
